In [ ]:
import sys
import torch
import logging
import numpy as np
from tqdm import tqdm
import multiprocessing
from datetime import datetime
import torchvision.transforms as T

import test
import util
import parser
import commons
import cosface_loss
import augmentations
from cosplace_model import cosplace_network
from datasets.test_dataset import TestDataset
from datasets.train_dataset import TrainDataset

In [ ]:
model = cosplace_network.GeoLocalizationNet(backbone="ResNet18", fc_output_dim=512, train_all_layers=False)

In [ ]:
model = model.to("cuda").train()

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
model_optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:

import os
import torch
import random
import logging
import numpy as np
from PIL import Image
from PIL import ImageFile
import torchvision.transforms as T
from collections import defaultdict

import datasets.dataset_utils as dataset_utils


ImageFile.LOAD_TRUNCATED_IMAGES = True


class TrainDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_folder, M=10, alpha=30, N=5, L=2,
                 current_group=0, min_images_per_class=10):
        """
        Parameters (please check our paper for a clearer explanation of the parameters).
        ----------
        args : args for data augmentation
        dataset_folder : str, the path of the folder with the train images.
        M : int, the length of the side of each cell in meters.
        alpha : int, size of each class in degrees.
        N : int, distance (M-wise) between two classes of the same group.
        L : int, distance (alpha-wise) between two classes of the same group.
        current_group : int, which one of the groups to consider.
        min_images_per_class : int, minimum number of image in a class.
        """
        super().__init__()
        self.M = M
        self.alpha = alpha
        self.N = N
        self.L = L
        self.current_group = current_group
        self.dataset_folder = dataset_folder
        
        # dataset_name should be either "processed", "small" or "raw", if you're using SF-XL
        dataset_name = os.path.basename(dataset_folder)
        filename = f"cache/{dataset_name}_M{M}_N{N}_alpha{alpha}_L{L}_mipc{min_images_per_class}.torch"
        if not os.path.exists(filename):
            os.makedirs("cache", exist_ok=True)
            logging.info(f"Cached dataset {filename} does not exist, I'll create it now.")
            self.initialize(dataset_folder, M, N, alpha, L, min_images_per_class, filename)
        elif current_group == 0:
            logging.info(f"Using cached dataset {filename}")
        
        classes_per_group, self.images_per_class = torch.load(filename)
        if current_group >= len(classes_per_group):
            raise ValueError(f"With this configuration there are only {len(classes_per_group)} " +
                             f"groups, therefore I can't create the {current_group}th group. " +
                             "You should reduce the number of groups by setting for example " +
                             f"'--groups_num {current_group}'")
        self.classes_ids = classes_per_group[current_group]
        
        
    
    @staticmethod
    def open_image(path):
        return Image.open(path).convert("RGB")
    
    def __getitem__(self, class_num):
        # This function takes as input the class_num instead of the index of
        # the image. This way each class is equally represented during training.
        
        class_id = self.classes_ids[class_num]
        # Pick a random image among those in this class.
        image_path = os.path.join(self.dataset_folder, random.choice(self.images_per_class[class_id]))
        
        try:
            pil_image = TrainDataset.open_image(image_path)
        except Exception as e:
            logging.info(f"ERROR image {image_path} couldn't be opened, it might be corrupted.")
            raise e
        
        tensor_image = T.functional.to_tensor(pil_image)
        assert tensor_image.shape == torch.Size([3, 512, 512]), \
            f"Image {image_path} should have shape [3, 512, 512] but has {tensor_image.shape}."
        
        
        
        return tensor_image, class_num, image_path
    
    def get_images_num(self):
        """Return the number of images within this group."""
        return sum([len(self.images_per_class[c]) for c in self.classes_ids])
    
    def __len__(self):
        """Return the number of classes within this group."""
        return len(self.classes_ids)
    
    @staticmethod
    def initialize(dataset_folder, M, N, alpha, L, min_images_per_class, filename):
        logging.debug(f"Searching training images in {dataset_folder}")
        
        images_paths = dataset_utils.read_images_paths(dataset_folder)
        logging.debug(f"Found {len(images_paths)} images")
        
        logging.debug("For each image, get its UTM east, UTM north and heading from its path")
        images_metadatas = [p.split("@") for p in images_paths]
        # field 1 is UTM east, field 2 is UTM north, field 9 is heading
        utmeast_utmnorth_heading = [(m[1], m[2], m[9]) for m in images_metadatas]
        utmeast_utmnorth_heading = np.array(utmeast_utmnorth_heading).astype(np.float64)
        
        logging.debug("For each image, get class and group to which it belongs")
        class_id__group_id = [TrainDataset.get__class_id__group_id(*m, M, alpha, N, L)
                              for m in utmeast_utmnorth_heading]
        
        logging.debug("Group together images belonging to the same class")
        images_per_class = defaultdict(list)
        for image_path, (class_id, _) in zip(images_paths, class_id__group_id):
            images_per_class[class_id].append(image_path)
        
        # Images_per_class is a dict where the key is class_id, and the value
        # is a list with the paths of images within that class.
        images_per_class = {k: v for k, v in images_per_class.items() if len(v) >= min_images_per_class}
        
        logging.debug("Group together classes belonging to the same group")
        # Classes_per_group is a dict where the key is group_id, and the value
        # is a list with the class_ids belonging to that group.
        classes_per_group = defaultdict(set)
        for class_id, group_id in class_id__group_id:
            if class_id not in images_per_class:
                continue  # Skip classes with too few images
            classes_per_group[group_id].add(class_id)
        
        # Convert classes_per_group to a list of lists.
        # Each sublist represents the classes within a group.
        classes_per_group = [list(c) for c in classes_per_group.values()]
        
        torch.save((classes_per_group, images_per_class), filename)
    
    @staticmethod
    def get__class_id__group_id(utm_east, utm_north, heading, M, alpha, N, L):
        """Return class_id and group_id for a given point.
            The class_id is a triplet (tuple) of UTM_east, UTM_north and
            heading (e.g. (396520, 4983800,120)).
            The group_id represents the group to which the class belongs
            (e.g. (0, 1, 0)), and it is between (0, 0, 0) and (N, N, L).
        """
        rounded_utm_east = int(utm_east // M * M)  # Rounded to nearest lower multiple of M
        rounded_utm_north = int(utm_north // M * M)
        rounded_heading = int(heading // alpha * alpha)
        
        class_id = (rounded_utm_east, rounded_utm_north, rounded_heading)
        # group_id goes from (0, 0, 0) to (N, N, L)
        group_id = (rounded_utm_east % (M * N) // M,
                    rounded_utm_north % (M * N) // M,
                    rounded_heading % (alpha * L) // alpha)
        return class_id, group_id


In [ ]:
groups = [TrainDataset("small/train/37.70") for n in range(1)]

In [ ]:
groups

In [ ]:
group.[0]

SyntaxError: invalid syntax (<ipython-input-8-a813fc42ec05>, line 1)

In [ ]:
groups[0]

In [ ]:
groups[0][0]

(tensor([[[0.4392, 0.4392, 0.4392,  ..., 0.5216, 0.5216, 0.5216],
          [0.4392, 0.4392, 0.4392,  ..., 0.5216, 0.5216, 0.5216],
          [0.4392, 0.4392, 0.4392,  ..., 0.5176, 0.5176, 0.5176],
          ...,
          [0.4392, 0.4392, 0.4353,  ..., 0.3333, 0.3333, 0.3333],
          [0.4431, 0.4431, 0.4392,  ..., 0.3333, 0.3333, 0.3333],
          [0.4431, 0.4471, 0.4431,  ..., 0.3333, 0.3333, 0.3333]],
 
         [[0.5020, 0.5020, 0.5020,  ..., 0.5882, 0.5882, 0.5882],
          [0.5020, 0.5020, 0.5020,  ..., 0.5882, 0.5882, 0.5882],
          [0.5020, 0.5020, 0.5020,  ..., 0.5843, 0.5843, 0.5843],
          ...,
          [0.3608, 0.3608, 0.3647,  ..., 0.3020, 0.3020, 0.3020],
          [0.3647, 0.3647, 0.3686,  ..., 0.3020, 0.3020, 0.3020],
          [0.3647, 0.3686, 0.3725,  ..., 0.3020, 0.3020, 0.3020]],
 
         [[0.5922, 0.5922, 0.5922,  ..., 0.6510, 0.6510, 0.6510],
          [0.5922, 0.5922, 0.5922,  ..., 0.6510, 0.6510, 0.6510],
          [0.5922, 0.5922, 0.5922,  ...,

In [ ]:
groups[0].get_images_num()

3070

In [ ]:
groups[0].__len__()

307

In [ ]:
groups[0][0][0]

tensor([[[0.5333, 0.5333, 0.5255,  ..., 0.4863, 0.4824, 0.4745],
         [0.5373, 0.5373, 0.5294,  ..., 0.4863, 0.4824, 0.4784],
         [0.5294, 0.5333, 0.5333,  ..., 0.4863, 0.4863, 0.4863],
         ...,
         [0.2902, 0.2902, 0.2902,  ..., 0.2745, 0.2745, 0.2745],
         [0.2902, 0.2902, 0.2902,  ..., 0.2667, 0.2667, 0.2667],
         [0.2902, 0.2902, 0.2902,  ..., 0.2667, 0.2667, 0.2667]],

        [[0.5451, 0.5451, 0.5490,  ..., 0.5451, 0.5412, 0.5333],
         [0.5490, 0.5490, 0.5529,  ..., 0.5451, 0.5412, 0.5373],
         [0.5529, 0.5569, 0.5569,  ..., 0.5451, 0.5451, 0.5451],
         ...,
         [0.2902, 0.2902, 0.2902,  ..., 0.2824, 0.2824, 0.2824],
         [0.2902, 0.2902, 0.2902,  ..., 0.2863, 0.2863, 0.2863],
         [0.2902, 0.2902, 0.2902,  ..., 0.2863, 0.2863, 0.2863]],

        [[0.6039, 0.6039, 0.6039,  ..., 0.6275, 0.6157, 0.6078],
         [0.6078, 0.6078, 0.6078,  ..., 0.6275, 0.6157, 0.6118],
         [0.6000, 0.6039, 0.6039,  ..., 0.6196, 0.6196, 0.

In [ ]:
groups[0][0][1]

0

In [ ]:
groups[0][0][2]

'small/train/37.70\\@0546909.62@4173407.94@10@S@037.70671@-122.46782@YvyuO8eAVSMSWqtUgPzxaQ@@0@@@@201409@@.jpg'

In [ ]:
groups[0][1][1]

1

In [ ]:
groups[0][2][1]

2

In [ ]:
classifiers = [cosface_loss.MarginCosineProduct(512, len(group)) for group in groups]
    classifiers_optimizers = [torch.optim.Adam(classifier.parameters(), lr=0.01) for classifier in classifiers]


IndentationError: unexpected indent (<ipython-input-18-2f1c2171619c>, line 2)

In [ ]:
classifiers = [cosface_loss.MarginCosineProduct(512, len(group)) for group in groups]
classifiers_optimizers = [torch.optim.Adam(classifier.parameters(), lr=0.01) for classifier in classifiers]


In [ ]:
logging.info(f"Using {len(groups)} groups")
logging.info(f"The {len(groups)} groups have respectively the following number of classes {[len(g) for g in groups]}")
logging.info(f"The {len(groups)} groups have respectively the following number of images {[g.get_images_num() for g in groups]}")

In [ ]:
print(f"Using {len(groups)} groups")
print(f"The {len(groups)} groups have respectively the following number of classes {[len(g) for g in groups]}")
print(f"The {len(groups)} groups have respectively the following number of images {[g.get_images_num() for g in groups]}")



Using 1 groups
The 1 groups have respectively the following number of classes [307]
The 1 groups have respectively the following number of images [3070]


In [ ]:
val_ds = TestDataset("small/val/database")
test_ds = TestDataset("small/test/database", queries_folder="small/test/queries_v1")

FileNotFoundError: Folder small/val/database/database does not exist

In [ ]:
val_ds = TestDataset("small/val")
test_ds = TestDataset("small/test", queries_folder="small/test/queries_v1")

FileNotFoundError: Folder small/test/small/test/queries_v1 does not exist

In [ ]:
val_ds = TestDataset("small/val/database")
test_ds = TestDataset("small/test/database", queries_folder="queries_v1")

FileNotFoundError: Folder small/val/database/database does not exist

In [ ]:
val_ds = TestDataset("small/val")
test_ds = TestDataset("small/test", queries_folder="queries_v1")

In [ ]:
test_ds

< test - #q: 1000; #db: 27191 >

In [ ]:
test_ds[0]

(tensor([[[1.3927, 1.3927, 1.3927,  ..., 1.1529, 1.1358, 1.1358],
          [1.3927, 1.3927, 1.3927,  ..., 1.1529, 1.1358, 1.1358],
          [1.3927, 1.3927, 1.3927,  ..., 1.1700, 1.1529, 1.1529],
          ...,
          [0.2796, 0.3823, 0.3481,  ..., 0.1939, 0.1939, 0.1939],
          [0.1254, 0.1597, 0.1768,  ..., 0.2111, 0.2111, 0.2111],
          [0.2967, 0.2111, 0.2624,  ..., 0.1426, 0.1254, 0.1083]],
 
         [[1.5532, 1.5532, 1.5532,  ..., 1.2906, 1.2731, 1.2731],
          [1.5532, 1.5532, 1.5532,  ..., 1.2906, 1.2731, 1.2731],
          [1.5532, 1.5532, 1.5532,  ..., 1.3081, 1.2906, 1.2906],
          ...,
          [0.3452, 0.4503, 0.4153,  ..., 0.2577, 0.2577, 0.2577],
          [0.1877, 0.2227, 0.2402,  ..., 0.2752, 0.2752, 0.2752],
          [0.3627, 0.2752, 0.3277,  ..., 0.2052, 0.1877, 0.1702]],
 
         [[1.7685, 1.7685, 1.7685,  ..., 1.4722, 1.4548, 1.4548],
          [1.7685, 1.7685, 1.7685,  ..., 1.4722, 1.4548, 1.4548],
          [1.7685, 1.7685, 1.7685,  ...,

In [ ]:
test_ds[0][1]

0

In [ ]:
test_ds.__class__()

TypeError: TestDataset.__init__() missing 1 required positional argument: 'dataset_folder'

In [ ]:
test_ds[1][0]

tensor([[[ 1.3242,  1.3242,  1.3242,  ...,  0.6563,  0.6563,  0.6563],
         [ 1.3242,  1.3242,  1.3242,  ...,  0.6734,  0.6563,  0.6563],
         [ 1.3242,  1.3242,  1.3242,  ...,  0.6734,  0.6734,  0.6563],
         ...,
         [-0.0287,  0.0227,  0.0569,  ...,  0.2796,  0.2967,  0.2967],
         [ 0.0398,  0.0912,  0.1254,  ...,  0.2624,  0.2967,  0.2967],
         [ 0.1768,  0.0741, -0.0116,  ...,  0.2624,  0.2796,  0.2967]],

        [[ 1.5007,  1.5007,  1.5007,  ...,  0.8704,  0.8704,  0.8704],
         [ 1.5007,  1.5007,  1.5007,  ...,  0.8880,  0.8704,  0.8704],
         [ 1.5007,  1.5007,  1.5007,  ...,  0.8880,  0.8880,  0.8704],
         ...,
         [ 0.1352,  0.1877,  0.2227,  ...,  0.2402,  0.2577,  0.2577],
         [ 0.2052,  0.2577,  0.2927,  ...,  0.2227,  0.2577,  0.2577],
         [ 0.3452,  0.2402,  0.1527,  ...,  0.2227,  0.2402,  0.2577]],

        [[ 1.7511,  1.7511,  1.7511,  ...,  1.1411,  1.1411,  1.1411],
         [ 1.7511,  1.7511,  1.7511,  ...,  1

In [ ]:
test_ds[1]

(tensor([[[ 1.3242,  1.3242,  1.3242,  ...,  0.6563,  0.6563,  0.6563],
          [ 1.3242,  1.3242,  1.3242,  ...,  0.6734,  0.6563,  0.6563],
          [ 1.3242,  1.3242,  1.3242,  ...,  0.6734,  0.6734,  0.6563],
          ...,
          [-0.0287,  0.0227,  0.0569,  ...,  0.2796,  0.2967,  0.2967],
          [ 0.0398,  0.0912,  0.1254,  ...,  0.2624,  0.2967,  0.2967],
          [ 0.1768,  0.0741, -0.0116,  ...,  0.2624,  0.2796,  0.2967]],
 
         [[ 1.5007,  1.5007,  1.5007,  ...,  0.8704,  0.8704,  0.8704],
          [ 1.5007,  1.5007,  1.5007,  ...,  0.8880,  0.8704,  0.8704],
          [ 1.5007,  1.5007,  1.5007,  ...,  0.8880,  0.8880,  0.8704],
          ...,
          [ 0.1352,  0.1877,  0.2227,  ...,  0.2402,  0.2577,  0.2577],
          [ 0.2052,  0.2577,  0.2927,  ...,  0.2227,  0.2577,  0.2577],
          [ 0.3452,  0.2402,  0.1527,  ...,  0.2227,  0.2402,  0.2577]],
 
         [[ 1.7511,  1.7511,  1.7511,  ...,  1.1411,  1.1411,  1.1411],
          [ 1.7511,  1.7511,

In [ ]:
test_ds[2]

(tensor([[[ 0.0398,  0.0398,  0.0398,  ..., -1.3302, -0.5253, -0.2856],
          [-0.0801, -0.0629, -0.0629,  ..., -1.3987, -1.3815,  0.0569],
          [-0.0287, -0.0116,  0.0056,  ..., -0.6794, -1.5357, -0.9020],
          ...,
          [ 0.6392,  0.6221,  0.6049,  ...,  0.0056, -0.0116, -0.0116],
          [ 0.6392,  0.6392,  0.6221,  ..., -0.0116, -0.0116, -0.0116],
          [ 0.6563,  0.6392,  0.6221,  ..., -0.0287, -0.0116,  0.0056]],
 
         [[ 0.1702,  0.1702,  0.1702,  ..., -1.3004, -0.4601, -0.2150],
          [ 0.0476,  0.0651,  0.0651,  ..., -1.3704, -1.3704,  0.1352],
          [ 0.1176,  0.1352,  0.1527,  ..., -0.6877, -1.5280, -0.8803],
          ...,
          [ 0.6954,  0.6779,  0.6604,  ...,  0.0651,  0.0476,  0.0476],
          [ 0.6954,  0.6954,  0.6779,  ...,  0.0476,  0.0476,  0.0476],
          [ 0.7129,  0.6954,  0.6779,  ...,  0.0301,  0.0476,  0.0651]],
 
         [[ 0.2522,  0.2522,  0.2522,  ..., -1.2293, -0.3578, -0.1138],
          [ 0.1302,  0.1476,

In [ ]:
test_ds[112]

(tensor([[[ 0.2111,  0.2111,  0.2111,  ...,  0.9646,  0.9646,  0.9646],
          [ 0.2111,  0.2111,  0.2111,  ...,  0.9646,  0.9646,  0.9646],
          [ 0.2111,  0.2111,  0.2111,  ...,  0.9646,  0.9646,  0.9646],
          ...,
          [-0.4397, -0.3369, -0.1999,  ..., -0.4911, -0.4397, -0.4397],
          [-0.3198, -0.2513, -0.1999,  ..., -0.4054, -0.4054, -0.4568],
          [-0.3027, -0.3027, -0.3027,  ..., -0.3541, -0.3883, -0.4739]],
 
         [[ 0.5203,  0.5203,  0.5203,  ...,  1.1506,  1.1506,  1.1506],
          [ 0.5203,  0.5203,  0.5203,  ...,  1.1506,  1.1506,  1.1506],
          [ 0.5203,  0.5203,  0.5203,  ...,  1.1506,  1.1506,  1.1506],
          ...,
          [-0.3901, -0.2850, -0.1450,  ..., -0.3375, -0.2850, -0.2850],
          [-0.2850, -0.2150, -0.1625,  ..., -0.2500, -0.1975, -0.2500],
          [-0.2675, -0.2675, -0.2675,  ..., -0.1975, -0.1800, -0.2675]],
 
         [[ 0.9145,  0.9145,  0.9145,  ...,  1.3502,  1.3502,  1.3502],
          [ 0.9145,  0.9145,

In [ ]:
classifiers

[MarginCosineProduct(in_features=512, out_features=307, s=30.0, m=0.4)]

In [ ]:
classifiers[0]

MarginCosineProduct(in_features=512, out_features=307, s=30.0, m=0.4)

In [ ]:
classifiers[0][0]

TypeError: 'MarginCosineProduct' object is not subscriptable

In [ ]:
current_group_num = 0 % 1

In [ ]:
classifiers[current_group_num] = classifiers[current_group_num].to("cuda")

In [ ]:
dataloader = commons.InfiniteDataLoader(groups[current_group_num],batch_size=32, shuffle=True, drop_last=True)
        

In [ ]:
dataloader_iterator = iter(dataloader)

In [ ]:
images, targets, _ = next(dataloader_iterator)

In [ ]:
images

tensor([[[[0.3020, 0.3020, 0.3020,  ..., 0.3569, 0.3569, 0.3569],
          [0.3020, 0.3020, 0.3020,  ..., 0.3569, 0.3569, 0.3569],
          [0.3020, 0.3020, 0.3020,  ..., 0.3569, 0.3569, 0.3569],
          ...,
          [0.6510, 0.8039, 0.7451,  ..., 0.4157, 0.5098, 0.5961],
          [0.6745, 0.7765, 0.7647,  ..., 0.4824, 0.5098, 0.5294],
          [0.6902, 0.7529, 0.7922,  ..., 0.5333, 0.4941, 0.4510]],

         [[0.5686, 0.5686, 0.5686,  ..., 0.5922, 0.5922, 0.5922],
          [0.5686, 0.5686, 0.5686,  ..., 0.5922, 0.5922, 0.5922],
          [0.5686, 0.5686, 0.5686,  ..., 0.5922, 0.5922, 0.5922],
          ...,
          [0.6157, 0.7765, 0.7176,  ..., 0.3843, 0.4706, 0.5529],
          [0.6392, 0.7490, 0.7373,  ..., 0.4510, 0.4706, 0.4863],
          [0.6549, 0.7255, 0.7647,  ..., 0.5020, 0.4549, 0.4078]],

         [[0.9569, 0.9569, 0.9569,  ..., 0.9294, 0.9294, 0.9294],
          [0.9569, 0.9569, 0.9569,  ..., 0.9294, 0.9294, 0.9294],
          [0.9569, 0.9569, 0.9569,  ..., 0

In [ ]:
images[0]

tensor([[[0.3020, 0.3020, 0.3020,  ..., 0.3569, 0.3569, 0.3569],
         [0.3020, 0.3020, 0.3020,  ..., 0.3569, 0.3569, 0.3569],
         [0.3020, 0.3020, 0.3020,  ..., 0.3569, 0.3569, 0.3569],
         ...,
         [0.6510, 0.8039, 0.7451,  ..., 0.4157, 0.5098, 0.5961],
         [0.6745, 0.7765, 0.7647,  ..., 0.4824, 0.5098, 0.5294],
         [0.6902, 0.7529, 0.7922,  ..., 0.5333, 0.4941, 0.4510]],

        [[0.5686, 0.5686, 0.5686,  ..., 0.5922, 0.5922, 0.5922],
         [0.5686, 0.5686, 0.5686,  ..., 0.5922, 0.5922, 0.5922],
         [0.5686, 0.5686, 0.5686,  ..., 0.5922, 0.5922, 0.5922],
         ...,
         [0.6157, 0.7765, 0.7176,  ..., 0.3843, 0.4706, 0.5529],
         [0.6392, 0.7490, 0.7373,  ..., 0.4510, 0.4706, 0.4863],
         [0.6549, 0.7255, 0.7647,  ..., 0.5020, 0.4549, 0.4078]],

        [[0.9569, 0.9569, 0.9569,  ..., 0.9294, 0.9294, 0.9294],
         [0.9569, 0.9569, 0.9569,  ..., 0.9294, 0.9294, 0.9294],
         [0.9569, 0.9569, 0.9569,  ..., 0.9294, 0.9294, 0.

In [ ]:
targets

tensor([290,  36, 210, 215,  34, 295, 213, 196,  90, 273,  58,  57, 181, 111,
        239, 100, 300,  71,  60,  83,  95,  38, 182, 132,  91,  42, 240, 205,
        130, 267, 257,  81])

In [ ]:
descriptors = model(images)
descriptors

RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

In [ ]:
images, targets = images.to(args.device), targets.to(args.device)

NameError: name 'args' is not defined

In [ ]:
images, targets = images.to("cuda"), targets.to("cuda")

In [ ]:
descriptors = model(images)
descriptors

tensor([[-0.0039,  0.0129, -0.0156,  ...,  0.0074, -0.0593, -0.0033],
        [ 0.0036,  0.0209,  0.0037,  ..., -0.0037, -0.0409,  0.0109],
        [ 0.0238, -0.0001, -0.0129,  ..., -0.0029, -0.0352, -0.0055],
        ...,
        [ 0.0062,  0.0111,  0.0145,  ..., -0.0125, -0.0477, -0.0017],
        [-0.0047,  0.0095,  0.0330,  ..., -0.0029, -0.0121,  0.0003],
        [ 0.0068,  0.0089, -0.0094,  ...,  0.0140, -0.0293,  0.0197]],
       device='cuda:0', grad_fn=<DivBackward0>)

In [ ]:
descriptors.shape

torch.Size([32, 512])

In [ ]:
output = classifiers[current_group_num](descriptors, targets)

In [ ]:
output

tensor([[ 0.3376,  0.7719,  0.8689,  ..., -1.3441, -0.1958, -2.1478],
        [ 0.5387,  0.2886,  0.9224,  ..., -1.0728, -0.3188, -2.5001],
        [ 0.0520,  0.4848,  1.0966,  ..., -1.3491, -0.5932, -3.0329],
        ...,
        [ 0.7871,  0.8286,  0.4308,  ..., -1.2490, -1.0938, -2.5780],
        [ 0.2440,  0.2623,  0.6850,  ..., -0.7178, -0.9610, -2.8339],
        [ 0.2775,  0.4426,  1.0020,  ..., -0.7868, -0.1863, -2.7882]],
       device='cuda:0', grad_fn=<MulBackward0>)

In [ ]:
output.shape

torch.Size([32, 307])

In [ ]:
model = model.eval()
    with torch.no_grad():
        logging.debug("Extracting database descriptors for evaluation/testing")
        database_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num)))
        database_dataloader = DataLoader(dataset=database_subset_ds,batch_size=32)
	all_descriptors = np.empty((len(eval_ds), args.fc_output_dim), dtype="float32")
        for images, indices in tqdm(database_dataloader, ncols=100):
            descriptors = model(images.to("cuda"))
            descriptors = descriptors.cpu().numpy()
            all_descriptors[indices.numpy(), :] = descriptors
        
        logging.debug("Extracting queries descriptors for evaluation/testing using batch size 1")
        queries_infer_batch_size = 1
        queries_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num, eval_ds.database_num+eval_ds.queries_num)))
        queries_dataloader = DataLoader(dataset=queries_subset_ds, batch_size=queries_infer_batch_size)
        for images, indices in tqdm(queries_dataloader, ncols=100):
            descriptors = model(images.to("cuda"))
            descriptors = descriptors.cpu().numpy()
            all_descriptors[indices.numpy(), :] = descriptors
    
    queries_descriptors = all_descriptors[eval_ds.database_num:]
    database_descriptors = all_descriptors[:eval_ds.database_num]
    
    # Use a kNN to find predictions
    faiss_index = faiss.IndexFlatL2(512)
    faiss_index.add(database_descriptors)
    del database_descriptors, all_descriptors
    
    logging.debug("Calculating recalls")
    _, predictions = faiss_index.search(queries_descriptors, max(RECALL_VALUES))

IndentationError: unexpected indent (<ipython-input-53-8e8137c7179f>, line 2)

In [ ]:
model = model.eval()
with torch.no_grad():
    logging.debug("Extracting database descriptors for evaluation/testing")
    database_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num)))
    database_dataloader = DataLoader(dataset=database_subset_ds,batch_size=32)
    all_descriptors = np.empty((len(eval_ds), 512), dtype="float32")
    for images, indices in tqdm(database_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors
    
    logging.debug("Extracting queries descriptors for evaluation/testing using batch size 1")
    queries_infer_batch_size = 1
    queries_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num, eval_ds.database_num+eval_ds.queries_num)))
    queries_dataloader = DataLoader(dataset=queries_subset_ds, batch_size=queries_infer_batch_size)
    for images, indices in tqdm(queries_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors

queries_descriptors = all_descriptors[eval_ds.database_num:]
database_descriptors = all_descriptors[:eval_ds.database_num]

# Use a kNN to find predictions
faiss_index = faiss.IndexFlatL2(512)
faiss_index.add(database_descriptors)
del database_descriptors, all_descriptors

logging.debug("Calculating recalls")
_, predictions = faiss_index.search(queries_descriptors, max(RECALL_VALUES))

NameError: name 'Subset' is not defined

In [ ]:

import faiss
import torch
import logging
import numpy as np
from tqdm import tqdm
from typing import Tuple
from argparse import Namespace
from torch.utils.data.dataset import Subset
from torch.utils.data import DataLoader, Dataset

import visualizations

In [ ]:
model = model.eval()
with torch.no_grad():
    logging.debug("Extracting database descriptors for evaluation/testing")
    database_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num)))
    database_dataloader = DataLoader(dataset=database_subset_ds,batch_size=32)
    all_descriptors = np.empty((len(eval_ds), 512), dtype="float32")
    for images, indices in tqdm(database_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors
    
    logging.debug("Extracting queries descriptors for evaluation/testing using batch size 1")
    queries_infer_batch_size = 1
    queries_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num, eval_ds.database_num+eval_ds.queries_num)))
    queries_dataloader = DataLoader(dataset=queries_subset_ds, batch_size=queries_infer_batch_size)
    for images, indices in tqdm(queries_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors

queries_descriptors = all_descriptors[eval_ds.database_num:]
database_descriptors = all_descriptors[:eval_ds.database_num]

# Use a kNN to find predictions
faiss_index = faiss.IndexFlatL2(512)
faiss_index.add(database_descriptors)
del database_descriptors, all_descriptors

logging.debug("Calculating recalls")
_, predictions = faiss_index.search(queries_descriptors, max(RECALL_VALUES))

NameError: name 'eval_ds' is not defined

In [ ]:
model = model.eval()
with torch.no_grad():
    logging.debug("Extracting database descriptors for evaluation/testing")
    database_subset_ds = Subset(test_ds, list(range(eval_ds.database_num)))
    database_dataloader = DataLoader(dataset=database_subset_ds,batch_size=32)
    all_descriptors = np.empty((len(eval_ds), 512), dtype="float32")
    for images, indices in tqdm(database_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors
    
    logging.debug("Extracting queries descriptors for evaluation/testing using batch size 1")
    queries_infer_batch_size = 1
    queries_subset_ds = Subset(eval_ds, list(range(eval_ds.database_num, eval_ds.database_num+eval_ds.queries_num)))
    queries_dataloader = DataLoader(dataset=queries_subset_ds, batch_size=queries_infer_batch_size)
    for images, indices in tqdm(queries_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors

queries_descriptors = all_descriptors[eval_ds.database_num:]
database_descriptors = all_descriptors[:eval_ds.database_num]

# Use a kNN to find predictions
faiss_index = faiss.IndexFlatL2(512)
faiss_index.add(database_descriptors)
del database_descriptors, all_descriptors

logging.debug("Calculating recalls")
_, predictions = faiss_index.search(queries_descriptors, max(RECALL_VALUES))

NameError: name 'eval_ds' is not defined

In [ ]:
RECALL_VALUES = [1, 5, 10, 20]
model = model.eval()
with torch.no_grad():
    logging.debug("Extracting database descriptors for evaluation/testing")
    database_subset_ds = Subset(test_ds, list(range(test_ds.database_num)))
    database_dataloader = DataLoader(dataset=database_subset_ds,batch_size=32)
    all_descriptors = np.empty((len(test_ds), 512), dtype="float32")
    for images, indices in tqdm(database_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors
    
    logging.debug("Extracting queries descriptors for evaluation/testing using batch size 1")
    queries_infer_batch_size = 1
    queries_subset_ds = Subset(test_ds, list(range(test_ds.database_num, test_ds.database_num+test_ds.queries_num)))
    queries_dataloader = DataLoader(dataset=queries_subset_ds, batch_size=queries_infer_batch_size)
    for images, indices in tqdm(queries_dataloader, ncols=100):
        descriptors = model(images.to("cuda"))
        descriptors = descriptors.cpu().numpy()
        all_descriptors[indices.numpy(), :] = descriptors

queries_descriptors = all_descriptors[test_ds.database_num:]
database_descriptors = all_descriptors[:test_ds.database_num]

# Use a kNN to find predictions
faiss_index = faiss.IndexFlatL2(512)
faiss_index.add(database_descriptors)
del database_descriptors, all_descriptors

logging.debug("Calculating recalls")
_, predictions = faiss_index.search(queries_descriptors, max(RECALL_VALUES))

100%|███████████████████████████████████████████████████████████| 1000/1000 [00:22<00:00, 43.88it/s]


In [ ]:
predictions

array([[21113, 26856, 25792, ..., 26855,   463,    56],
       [ 9753,  9069, 12625, ..., 25997, 21504, 10206],
       [ 3644, 15260, 24263, ..., 18770, 25598,  2515],
       ...,
       [14394, 16033, 19874, ..., 24544, 22403, 24499],
       [ 1620, 27084, 18274, ..., 10684, 21269, 11718],
       [ 2433,  4935, 20018, ..., 19512, 15262, 18891]], dtype=int64)

In [ ]:
predictions.shape

(1000, 20)

In [ ]:
predictions[0]

array([21113, 26856, 25792, 24654, 26862, 14510,  4439, 16163, 26887,
         271, 25207,  3647,   577,    64, 26352, 24637,  4440, 26855,
         463,    56], dtype=int64)